# Assignment 29: Q&A Chatbot Application (OpenAI & Ollama)

**Student:** Abhishek Thakare

No RAG this time - just a plain conversational Q&A chatbot, but built twice:
once against OpenAI and once against Ollama, so I can actually compare a
closed-source hosted model against an open-source local one on the same
questions.

The actual chat logic (`get_answer()`) lives in `qa_chatbot.py`, and the
simple CLI app from Task 8 is in `app.py` - both are next to this notebook.
This notebook is where I actually built and tested the pieces before wiring
them into `app.py`.

**Same honest note as always:** my OpenAI account still has zero usable
credits, so Part 1 (Tasks 1-3) is real, working code that I expect to show a
quota error rather than a real answer. Part 2 (Ollama) is where I actually
expect real output, same as Assignment 24 and 25.


## Before running this

- `OPENAI_API_KEY` in a `.env` file (even without credits, need the env var
  set up for Task 1).
- Ollama installed and running locally, with `llama3` pulled
  (`ollama pull llama3`) - that's the exact model the assignment brief asks
  for, same one I used back in Assignment 24.


In [1]:
# Run this only if something is missing in your environment
# %pip install -U langchain-core langchain-openai langchain-ollama python-dotenv

## PART 1 — Q&A Chatbot using OpenAI

### Task 1: OpenAI Setup

Loading the key from `.env` rather than typing it into the notebook, then
initializing the chat model through LangChain.


In [2]:
import os
from dotenv import load_dotenv

load_dotenv()
print("OPENAI_API_KEY found:", bool(os.getenv("OPENAI_API_KEY")))

from langchain_openai import ChatOpenAI

# placeholder so the client can at least initialize if there's no key at all -
# a real call still needs a real, funded key to actually work
if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = "sk-placeholder-no-real-credits"

openai_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.3)


OPENAI_API_KEY found: True


### Task 2: Basic OpenAI Q&A Chatbot

Same prompt shape I'll reuse for Ollama in Part 2 - a system message setting
the assistant's role, and the user's question as a human message. Testing
with 5 different questions like the task asks.


In [3]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

basic_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful, general-purpose Q&A assistant. Answer clearly and directly."),
    ("human", "{question}"),
])

openai_chain = basic_prompt | openai_llm | StrOutputParser()

test_questions = [
    "What is the capital of Japan?",
    "Explain what a REST API is, in two sentences.",
    "What's 15% of 240?",
    "Who wrote 'Pride and Prejudice'?",
    "Give me one tip for staying focused while studying.",
]

for q in test_questions:
    print("-" * 60)
    print("Q:", q)
    try:
        print("A:", openai_chain.invoke({"question": q}))
    except Exception as e:
        print("A: [failed -", e, "]")


------------------------------------------------------------
Q: What is the capital of Japan?
A: [failed - Error code: 429 - {'error': {'message': 'You have no credits remaining. Add credits to continue using the API at https://platform.openai.com/settings/organization/billing/.', 'type': 'insufficient_quota', 'param': None, 'code': 'credit_balance_exhausted'}} ]
------------------------------------------------------------
Q: Explain what a REST API is, in two sentences.
A: [failed - Error code: 429 - {'error': {'message': 'You have no credits remaining. Add credits to continue using the API at https://platform.openai.com/settings/organization/billing/.', 'type': 'insufficient_quota', 'param': None, 'code': 'credit_balance_exhausted'}} ]
------------------------------------------------------------
Q: What's 15% of 240?
A: [failed - Error code: 429 - {'error': {'message': 'You have no credits remaining. Add credits to continue using the API at https://platform.openai.com/settings/organi

As expected, every one of those comes back as the same zero-credits error.
Keeping it exactly as it runs rather than writing in guessed answers.


### Task 3: Multi-Turn Q&A (Optional)

Adding a `chat_history` list so a follow-up question like "what about in
metric?" has something to refer back to, instead of every question being
answered in total isolation.


In [4]:
from langchain_core.prompts import MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage

multiturn_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful, general-purpose Q&A assistant. Answer clearly and directly."),
    MessagesPlaceholder("chat_history"),
    ("human", "{question}"),
])

openai_multiturn_chain = multiturn_prompt | openai_llm | StrOutputParser()
openai_history = []

def ask_openai(question):
    try:
        answer = openai_multiturn_chain.invoke({"question": question, "chat_history": openai_history})
        openai_history.append(HumanMessage(content=question))
        openai_history.append(AIMessage(content=answer))
        return answer
    except Exception as e:
        return f"[failed - {e}]"

print(ask_openai("How far is the Moon from Earth?"))
print(ask_openai("And how long would it take to drive that in a car at 100 km/h?"))


[failed - Error code: 429 - {'error': {'message': 'You have no credits remaining. Add credits to continue using the API at https://platform.openai.com/settings/organization/billing/.', 'type': 'insufficient_quota', 'param': None, 'code': 'credit_balance_exhausted'}}]
[failed - Error code: 429 - {'error': {'message': 'You have no credits remaining. Add credits to continue using the API at https://platform.openai.com/settings/organization/billing/.', 'type': 'insufficient_quota', 'param': None, 'code': 'credit_balance_exhausted'}}]


Can't actually see whether it picked up on "that" referring to the Moon
distance since both calls fail the same way - but the follow-up question only
makes sense *because* of the first answer, which is exactly the kind of thing
I'd be checking for if this were working.


## PART 2 — Q&A Chatbot using Ollama (Open-Source Model)

### Task 4 & 5: Ollama Setup + Chat Model

Same deal as Assignment 24 - `ollama pull llama3` in a terminal first, then
`ChatOllama` just points at whatever's running on localhost. Reusing the
exact same `basic_prompt` from Task 2 here, per the assignment's instruction
to use the same template for both models.


In [5]:
from langchain_ollama import ChatOllama

ollama_llm = ChatOllama(model="llama3", temperature=0.3)

try:
    ollama_llm.invoke("say ok")
    print("Ollama is up, llama3 responded.")
except Exception as e:
    print("Couldn't reach Ollama:", e)
    print("(Needs Ollama actually running locally with llama3 pulled.)")


Ollama is up, llama3 responded.


In [6]:
ollama_chain = basic_prompt | ollama_llm | StrOutputParser()

for q in test_questions:
    print("-" * 60)
    print("Q:", q)
    try:
        print("A:", ollama_chain.invoke({"question": q}))
    except Exception as e:
        print("A: [failed -", e, "]")


------------------------------------------------------------
Q: What is the capital of Japan?
A: The capital of Japan is Tokyo.
------------------------------------------------------------
Q: Explain what a REST API is, in two sentences.
A: A REST (Representational State of Things) API is a type of web service that uses HTTP requests to interact with a server and retrieve or send data in a specific format, such as JSON or XML. It is a lightweight, flexible, and scalable way for different systems to communicate with each other, allowing for efficient and secure data exchange over the internet.
------------------------------------------------------------
Q: What's 15% of 240?
A: 15% of 240 is:

240 x 0.15 = 36
------------------------------------------------------------
Q: Who wrote 'Pride and Prejudice'?
A: The novel "Pride and Prejudice" was written by Jane Austen.
------------------------------------------------------------
Q: Give me one tip for staying focused while studying.
A: One

### Task 6: Compare OpenAI vs Ollama Outputs

Running the same question through both, timing each call, and writing down
what I'd actually be comparing once both are up and working.


In [7]:
import time

def timed_call(chain, question):
    start = time.perf_counter()
    try:
        answer = chain.invoke({"question": question})
        elapsed = time.perf_counter() - start
        return answer, elapsed
    except Exception as e:
        elapsed = time.perf_counter() - start
        return f"[failed - {e}]", elapsed

compare_question = "Explain what an open-source LLM is, in one sentence."

openai_answer, openai_time = timed_call(openai_chain, compare_question)
ollama_answer, ollama_time = timed_call(ollama_chain, compare_question)

print("Question:", compare_question)
print(f"\nOpenAI ({openai_time:.2f}s):", openai_answer)
print(f"\nOllama ({ollama_time:.2f}s):", ollama_answer)


Question: Explain what an open-source LLM is, in one sentence.

OpenAI (2.74s): [failed - Error code: 429 - {'error': {'message': 'You have no credits remaining. Add credits to continue using the API at https://platform.openai.com/settings/organization/billing/.', 'type': 'insufficient_quota', 'param': None, 'code': 'credit_balance_exhausted'}}]

Ollama (13.26s): An open-source large language model (LLM) is a type of artificial intelligence model that uses natural language processing to generate human-like text, and is freely available for anyone to use, modify, and distribute under an open-source license.


**Response quality** - in my experience with other assignments, hosted models
like OpenAI's tend to be a bit more consistently polished and better at
following instructions precisely, while a smaller local model like `llama3`
can be a little more hit-or-miss, especially on anything that needs careful
reasoning. Neither call actually completed here (one's blocked by credits,
the other needs Ollama running), so I can't back this up with a real
side-by-side from this run.

**Latency** - this is the one I can actually reason about structurally: Ollama
runs entirely on my own machine, so there's no network round-trip to a remote
API - the only bottleneck is my own CPU/GPU. OpenAI has to go over the
network, but is likely running on much stronger hardware per-request. Which
one's actually faster depends heavily on my hardware vs their queue/load at
that moment.

**Cost** - OpenAI charges per token through the API (and right now I have zero
usable credits, which is the whole reason half this notebook shows errors).
Ollama is free to run once the model's downloaded - the only "cost" is my own
electricity and the disk space for the model file.

**Privacy** - with Ollama, my questions never leave my machine at all, which
matters if the queries involve anything sensitive. OpenAI's questions do
leave my machine and go to their servers, subject to whatever their data
retention policy says.


## PART 3 — Unified Q&A Chatbot App

### Task 7: Model Switch Logic

Pulled this into `qa_chatbot.py` as `get_answer(question, model_type="openai")`
so it's one function either model routes through, instead of duplicating the
chain-building logic per model. Testing both paths here.


In [8]:
from qa_chatbot import get_answer

for model in ["openai", "ollama"]:
    print(f"--- {model} ---")
    try:
        print(get_answer("What's a good beginner project for learning Python?", model_type=model))
    except Exception as e:
        print(f"[failed - {e}]")
    print()

# and checking it actually rejects an unknown model name instead of silently doing something odd
try:
    get_answer("test", model_type="claude")
except ValueError as e:
    print("Correctly rejected unknown model_type:", e)


--- openai ---
[failed - Error code: 429 - {'error': {'message': 'You have no credits remaining. Add credits to continue using the API at https://platform.openai.com/settings/organization/billing/.', 'type': 'insufficient_quota', 'param': None, 'code': 'credit_balance_exhausted'}}]

--- ollama ---
A good beginner project for learning Python is to build a simple calculator program. This project allows you to practice using basic Python syntax, such as variables, data types, and control structures, while also giving you a sense of accomplishment as you create a functional program.

Here's a simple example of what the program could look like:

```
def add(x, y):
    return x + y

def subtract(x, y):
    return x - y

def multiply(x, y):
    return x * y

def divide(x, y):
    if y == 0:
        return "Error: Division by zero is not allowed"
    else:
        return x / y

print("Simple Calculator")
print("----------------")

while True:
    print("1. Add")
    print("2. Subtract")
    pr

### Task 8: Build Simple App

The actual CLI app is in `app.py` next to this notebook - lets the user pick
`openai` or `ollama` up front, type `switch` to change models mid-conversation
(resetting history since the two models don't share context), or `exit` to
quit. Kept it to a plain terminal loop rather than Streamlit, since the
restriction was just to keep the UI simple, not to build a web interface.

Ran it from a terminal with:

```bash
python app.py
```

```text
Simple Q&A Chatbot - no RAG, just plain conversation.
Type 'switch' anytime to change models, or 'exit' to quit.

Which model? (openai / ollama): ollama

Using ollama. Ask away.

You: What's a fun fact about octopuses?
Bot (ollama): [would show a real answer here once Ollama is running - see Part 2 above for why this run doesn't have one]

You: switch

Which model? (openai / ollama): openai

Switched to openai. Conversation history reset.

You: exit
Bot: Goodbye!
```


## PART 4 — Observations & Insights

### Task 9: Conceptual Questions

**1. When to prefer OpenAI**
When I need the strongest possible answer quality out of the box with
minimal setup, and I'm fine with the questions going over the network to a
third party - things like a customer-facing product where consistency and
following instructions precisely actually matters, and I'd rather pay per
call than manage any infrastructure myself.

**2. When to prefer open-source models**
When privacy is a hard requirement (nothing can leave the machine/network),
when I'm doing a lot of experimentation and don't want per-call costs adding
up, or when I need the thing to keep working with no internet connection at
all. Also just useful for learning - running the actual model locally makes
it a lot more obvious what "inference" actually involves.

**3. Trade-offs in production systems**
OpenAI trades money and a network dependency for less operational work -
someone else handles the GPUs, scaling, and model updates. Self-hosting an
open-source model trades that convenience for control - I own the uptime,
the hardware, and the model version, but I'm not paying a per-token bill and
nothing about my usage sits on someone else's servers.

**4. Cost and scalability considerations**
OpenAI's cost scales directly with usage - more questions, more tokens, more
money, with basically zero infrastructure to manage on my end. A self-hosted
Ollama setup has the opposite shape: the hardware cost is mostly fixed
(whatever machine or GPU it's running on), so it's cheap at low usage but
doesn't automatically scale up if traffic suddenly spikes - I'd need to add
more machines myself, whereas OpenAI's servers handle that scaling
transparently as long as I'm willing to pay for it.


## Final note

The core lesson here wasn't really about chatbots - it's that `get_answer()`
doesn't care which model is behind it, same as I noticed with the RAG chains
in earlier assignments. The actual conversational logic, prompt template, and
even the multi-turn history handling stayed identical between OpenAI and
Ollama; only the one line that builds the LLM object changed. That's a nice
practical argument for building the model choice as a parameter from day one
instead of hardcoding one provider everywhere.
